# Phase VIB-STAGGER: Architectural Fixes for the Negative Transfer

**Defense / ICTAI follow-up.** Two reviewer-style critiques of the original GLADA design:

1. **VIB placement is wrong.** Currently the VIB compresses the *pooled* representation, *after* the per-group discriminators have already operated on uncompressed `group_outputs`. The discriminators may be cannibalizing each other's gradients on raw, redundant per-group features. Fix: instantiate one VIB per group and compress *before* each discriminator sees its group's features.

2. **All 5 signals fire simultaneously when the backbone unfreezes.** Phase 2 begins with spoof + per-group GLADA (×G) + global VIB KL + speaker GRL + CKA all active at full strength against a freshly-trainable backbone. This is the gradient-shock scenario discussed in `phase2_adv_warmup_epochs`. Fix: introduce one auxiliary signal at a time over Phase 2 instead of all-at-once.

**Configs trained (3, all on `wavlm-base-plus`, seed 42):**
1. `per_group_vib` — fix 1 only (per-group VIB, normal schedule)
2. `staggered`     — fix 2 only (global VIB unchanged, signals introduced sequentially)
3. `per_group_vib_staggered` — both fixes combined

Baseline `full_glada` at 19.71% ITW EER comes from `Coding/results.json`; no need to retrain it.

**Decision rule:** any of the 3 variants beating 14.89% (`no_glada` baseline) means the negative transfer is *fixable*, which is a substantive contribution. Beating 12.98% (`no_cka`, the best wavlm-bp ablation) makes GLADA competitive with the strongest simpler variant. Failing to beat 19.71% means neither fix helps and the negative-transfer claim is robust to these specific architectural changes.


## 1. Environment Setup


In [ ]:
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
else:
    raise RuntimeError('No GPU detected')


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB (85.1 GB)


## 2. Copy GLADA Package to Local Runtime


In [ ]:
import shutil, os, sys

# Probe candidate paths for the package directory that contains __init__.py.
# The actual package dir must directly contain __init__.py; we then copy it
# to /content/glada and add /content to sys.path so `import glada` works.
CANDIDATES = [
    "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code/glada",
    '/content/drive/MyDrive/Jakob/glada_pkg/glada',
    '/content/drive/MyDrive/Jakob/glada_pkg',
    '/content/drive/MyDrive/Jakob/glada',
]
DRIVE_PKG = next((c for c in CANDIDATES if os.path.exists(os.path.join(c, '__init__.py'))), None)
if DRIVE_PKG is None:
    raise FileNotFoundError(
        'No glada/__init__.py found in candidates. Run:\n'
        "  !find /content/drive -name '__init__.py' -path '*/glada/*' 2>/dev/null | head\n"
        'and add the parent directory to CANDIDATES above.'
    )

LOCAL_PKG = '/content/glada'
if os.path.exists(LOCAL_PKG): shutil.rmtree(LOCAL_PKG)
shutil.copytree(DRIVE_PKG, LOCAL_PKG)

# Drop any stale glada module cache (in case a prior cell already imported)
for mod in list(sys.modules):
    if mod == 'glada' or mod.startswith('glada.'): del sys.modules[mod]
sys.path = [p for p in sys.path if 'glada' not in p]
if '/content' not in sys.path: sys.path.insert(0, '/content')
print(f'Package source: {DRIVE_PKG}\nCopied to:      {LOCAL_PKG}')


Package source: /content/drive/MyDrive/Jakob/glada_pkg/glada
Copied to:      /content/glada


## 3. Datasets (ASVspoof 2019 LA + ITW)


In [ ]:
import tarfile

def extract_dataset(tar_path, extract_to, name):
    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f'{name}: already extracted to {extract_to}'); return
    os.makedirs(extract_to, exist_ok=True)
    print(f'{name}: extracting {tar_path} -> {extract_to} ...')
    with tarfile.open(tar_path) as t: t.extractall(extract_to)
    print(f'{name}: done.')

extract_dataset('/content/drive/MyDrive/Jakob/Datasets/LA.tar', '/content/data/asvspoof', 'ASVspoof 2019 LA')
extract_dataset('/content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar', '/content/data/itw', 'In-the-Wild')

ASV_PROTO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ASV_AUDIO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_train/flac'
ASV_EVAL_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
ASV_EVAL_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
ITW_META       = '/content/data/itw/release_in_the_wild/meta.csv'
ITW_AUDIO      = '/content/data/itw/release_in_the_wild'
for p in [ASV_PROTO, ASV_AUDIO, ASV_EVAL_PROTO, ITW_META]:
    assert os.path.exists(p), f'Missing: {p}'
print('Datasets verified.')


ASVspoof 2019 LA: extracting /content/drive/MyDrive/Jakob/Datasets/LA.tar -> /content/data/asvspoof ...


/tmp/ipykernel_452/3241063313.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(tar_path) as t: t.extractall(extract_to)


ASVspoof 2019 LA: done.
In-the-Wild: extracting /content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar -> /content/data/itw ...
In-the-Wild: done.
Datasets verified.


## 4. Imports


In [ ]:
from glada import GLADAConfig, BACKBONE_REGISTRY, set_seed
from glada.datasets import (
    ASVspoofDatasetWithSpeaker,
    ASVspoofEvalDataset,
    InTheWildDataset,
    create_dataloaders,
)
from glada.augmentation import RawBoost
from glada.models import GLADAModel
from glada.models.vib import VIBBottleneck
from glada.models.grl import GradientReversal
from glada.models.discriminators import GroupedLayerDomainDiscriminators
from glada.trainers import GLADATrainer

import copy, gc, json, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

print('Imports OK')


Imports OK


## 5. INLINE: `PerGroupVIBModel`

Subclasses `GLADAModel`. Adds K `VIBBottleneck` modules (one per group), each compressing a `(B, 2D)` group ASP output to `(B, bottleneck_dim)`. The per-group discriminator then receives the compressed `z_k` rather than the raw `g_k`. Hierarchical pooling and the global VIB are bypassed: the spoof classifier sees the concatenated `[z_1; z_2; ...; z_K]` projected to `bottleneck_dim`.

Why this might help: the original architecture lets per-group discriminators race over high-dimensional, non-orthogonal features, with the global VIB acting too late to clean up the resulting representation. Per-group VIB acts as a per-discriminator information-bottleneck regularizer in the spirit of IB-CAAN, applied G times.


In [ ]:
class PerGroupVIBModel(GLADAModel):
    """GLADAModel with one VIBBottleneck per group, applied BEFORE the
    per-group discriminators. The hierarchical pool + global VIB are
    replaced by a simple linear projection over the concatenated
    per-group compressed representations.

    Architecturally:
        h_i  = LayerNorm_i(backbone(x))                 (per-layer norm, unchanged)
        g_k  = ASP(group_attn(h_i for i in group_k))   (per-group ASP, unchanged)
        z_k, kl_k = VIB_k(g_k)                          (NEW: per-group VIB)
        D_k(GRL(z_k))                                   (disc on COMPRESSED rep)
        z = LinearProj(concat(z_1, ..., z_K))           (pool over compressed)
        spoof_logit = Classifier(z)                     (no global VIB)
        speaker_logit = SpeakerAdv(GRL(z))              (unchanged target)
    """

    def __init__(self, config, num_speakers=None):
        super().__init__(config, num_speakers=num_speakers)
        K = config.num_groups
        D = config.hidden_dim
        bd = config.bottleneck_dim
        # Each group's ASP output has dim 2D (mean + std). Compress to bottleneck_dim.
        self.group_vibs = nn.ModuleList([
            VIBBottleneck(2 * D, bd) for _ in range(K)
        ])
        # Project concatenated per-group z's back to bottleneck_dim for the classifier.
        self.group_compressed_proj = nn.Linear(K * bd, bd)
        # Replace per-group discriminator input dim from 2D to bottleneck_dim.
        # The original GroupedLayerDomainDiscriminators sized its input to 2D;
        # we rebuild it sized to bd so the discriminators consume compressed reps.
        if self.glada_disc is not None:
            self.glada_disc = GroupedLayerDomainDiscriminators(
                num_groups=K,
                input_dim=bd,
                hidden_dim=config.glada_disc_hidden,
            )
        # The original global VIB and hierarchical pool become unused; we keep
        # them as identity/no-ops so checkpoints and downstream code don't break,
        # but the forward below routes around them.

    def forward(self, input_values, attention_mask=None):
        # Reuse the parent forward up through group_outputs computation,
        # then divert. Easier: replicate the relevant sections.
        backbone_frozen = not any(p.requires_grad for p in self.backbone.parameters())
        if backbone_frozen:
            with torch.no_grad():
                outputs = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        else:
            outputs = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        hidden_states = list(outputs.hidden_states)

        # Frame mask
        if attention_mask is not None:
            input_lengths = attention_mask.sum(dim=1)
            output_lengths = ((input_lengths - 1) // 320) + 1
            T = hidden_states[0].shape[1]
            frame_mask = (torch.arange(T, device=attention_mask.device).unsqueeze(0) < output_lengths.unsqueeze(1))
        else:
            frame_mask = None

        # Per-layer norm
        hidden_states = self.layer_norms(hidden_states)

        # Hierarchical pooling produces (pooled, cka_loss, group_outputs).
        # We discard `pooled` and use only group_outputs + cka_loss.
        _, cka_loss, group_outputs = self.pooling(hidden_states, frame_mask=frame_mask)
        if not self.config.use_cka_loss:
            cka_loss = torch.tensor(0.0, device=group_outputs[0].device)

        # Per-group VIB: compress each g_k -> z_k, accumulate KL.
        z_groups = []
        kl_total = torch.tensor(0.0, device=group_outputs[0].device)
        for k, g_k in enumerate(group_outputs):
            z_k, kl_k = self.group_vibs[k](g_k, training=self.training)
            z_groups.append(z_k)
            kl_total = kl_total + kl_k
        kl_total = kl_total / max(len(group_outputs), 1)  # mean across groups

        # Project concatenated compressed reps to bottleneck_dim for classifier/speaker.
        z_concat = torch.cat(z_groups, dim=-1)
        z = self.group_compressed_proj(z_concat)

        spoof_logits = self.task_classifier(z)
        result = {
            'spoof_logits': spoof_logits,
            'kl_loss': kl_total,
            'cka_loss': cka_loss,
            'features': z,
        }

        # Per-group discriminators on COMPRESSED reps (the whole point of this experiment).
        if self.glada_disc is not None:
            result['glada_domain_logits'] = self.glada_disc(z_groups, self.domain_grl_alpha)
            result['group_outputs'] = z_groups

        # Optional global domain disc on z (for the use_global_domain_disc baseline).
        if self.domain_disc is not None:
            z_domain = GradientReversal.apply(z, self.domain_grl_alpha)
            result['domain_logits'] = self.domain_disc(z_domain)

        # Speaker adversary on z.
        if self.speaker_adversary is not None:
            z_speaker = GradientReversal.apply(z, self.speaker_grl_alpha)
            result['speaker_logits'] = self.speaker_adversary(z_speaker)

        return result

print('PerGroupVIBModel defined.')


PerGroupVIBModel defined.


## 6. INLINE: `StaggeredAlphaTrainer`

Subclasses `GLADATrainer` and mutates `self.config` at the start of each epoch to gate auxiliary signals according to a schedule. Phase 1 is unchanged; in Phase 2 only `spoof` is active at the start, and components are introduced one at a time. The schedule below introduces a new auxiliary signal every ~3 epochs.


In [ ]:
STAGGER_SCHEDULE = {
    # Phase 2 epoch -> dict of config flags to ENABLE at that epoch.
    # Anything not in the schedule yet is disabled. Once enabled, stays on.
    0: {'use_vib': False, 'use_glada': False, 'use_global_domain_disc': False,
        'use_speaker_adversary': False, 'use_cka_loss': False},
    2: {'use_vib': True},                 # Add VIB compression
    5: {'use_glada': True},               # Add per-group adversarial DA
    8: {'use_speaker_adversary': True},   # Add speaker adversary
    11: {'use_cka_loss': True},           # Finally add CKA diversity
}

class StaggeredAlphaTrainer(GLADATrainer):
    """Mutates self.config at the start of each Phase 2 epoch to enable
    auxiliary signals according to STAGGER_SCHEDULE. The base trainer's
    train_epoch reads self.config.use_X per batch, so flag mutation
    propagates without further surgery.
    """

    def __init__(self, *args, stagger_schedule=None, target_flags=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._stagger_schedule = stagger_schedule or STAGGER_SCHEDULE
        # Snapshot the user's intended FINAL flag values so we can restore them
        # at the appropriate stagger step. target_flags lets the caller specify
        # which flags were meant to be on at all (e.g., for the per_group_vib_staggered
        # config, use_cka may stay False even after epoch 11).
        self._target_flags = target_flags or {
            'use_vib': self.config.use_vib,
            'use_glada': self.config.use_glada,
            'use_global_domain_disc': self.config.use_global_domain_disc,
            'use_speaker_adversary': self.config.use_speaker_adversary,
            'use_cka_loss': self.config.use_cka_loss,
        }

    def train_epoch(self, epoch):
        # In Phase 1, leave all flags as configured (heads-only training,
        # auxiliary signals on the heads themselves are fine).
        if self.current_phase == 2:
            p2_epoch = epoch - self.config.phase1_epochs
            # Find the latest schedule entry <= p2_epoch.
            for sched_epoch in sorted(self._stagger_schedule.keys()):
                if sched_epoch > p2_epoch:
                    break
                for flag, val in self._stagger_schedule[sched_epoch].items():
                    # Don't enable a flag the user explicitly disabled in target_flags.
                    if val and not self._target_flags.get(flag, True):
                        continue
                    setattr(self.config, flag, val)
            print(f'  [STAGGER] P2 epoch {p2_epoch}: '
                  f'vib={self.config.use_vib} '
                  f'glada={self.config.use_glada} '
                  f'spk={self.config.use_speaker_adversary} '
                  f'cka={self.config.use_cka_loss}')
        return super().train_epoch(epoch)

print('StaggeredAlphaTrainer defined.')


StaggeredAlphaTrainer defined.


## 7. Configurations


In [ ]:
# All three variants run on wavlm-base-plus, seed 42, same epoch budget as the
# original ablation (3 Phase 1 + 17 Phase 2 = ~5h on A100).
USE_FULL_EPOCHS = False
PHASE1 = 5  if USE_FULL_EPOCHS else 3
PHASE2 = 45 if USE_FULL_EPOCHS else 17

DRIVE_LOG_DIR = '/content/drive/MyDrive/Jakob/Experiments/vib_stagger'
os.makedirs(DRIVE_LOG_DIR, exist_ok=True)

def make_config(name, backbone='wavlm-base-plus', seed=42):
    cfg = GLADAConfig(
        experiment_name=name,
        backbone=backbone,
        seed=seed,
        batch_size=64,
        num_workers=0,
        phase1_epochs=PHASE1,
        phase2_epochs=PHASE2,
        drive_log_dir=DRIVE_LOG_DIR,
        run_id=time.strftime('%Y%m%d_%H%M%S'),
    )
    return cfg

VARIANTS = {
    # Variant 1: per-group VIB, normal schedule (no staggering)
    'per_group_vib':            {'model_class': 'PerGroupVIBModel', 'staggered': False},
    # Variant 2: original model, staggered schedule
    'staggered':                {'model_class': 'GLADAModel',       'staggered': True},
    # Variant 3: both fixes
    'per_group_vib_staggered':  {'model_class': 'PerGroupVIBModel', 'staggered': True},
}
for name, spec in VARIANTS.items():
    print(f'  {name:<30s} model={spec["model_class"]:<22s} stagger={spec["staggered"]}')


  per_group_vib                  model=PerGroupVIBModel       stagger=False
  staggered                      model=GLADAModel             stagger=True
  per_group_vib_staggered        model=PerGroupVIBModel       stagger=True


## 8. Datasets (shared across all 3 variants)


In [ ]:
base_for_data = make_config('_data_setup')
rawboost = RawBoost

source_ds = ASVspoofDatasetWithSpeaker(
    protocol_path=ASV_PROTO,
    audio_dir=ASV_AUDIO,
    config=base_for_data,
    augmenter=rawboost,  # RawBoost only — matches original ablation
)
target_ds = InTheWildDataset(
    meta_csv_path=ITW_META, audio_dir=ITW_AUDIO,
    config=base_for_data, use_labels=False,
)
eval_ds = InTheWildDataset(
    meta_csv_path=ITW_META, audio_dir=ITW_AUDIO,
    config=base_for_data, use_labels=True,
)
extra_eval = {
    '2019_LA': ASVspoofEvalDataset(
        protocol_path=ASV_EVAL_PROTO, audio_dir=ASV_EVAL_AUDIO, config=base_for_data,
    ),
}
num_speakers = len(source_ds.speaker_to_id) if hasattr(source_ds, 'speaker_to_id') else None
print(f'Source: {len(source_ds):,} ({num_speakers} speakers)')
print(f'Target: {len(target_ds):,}, Eval: {len(eval_ds):,}, 2019 LA: {len(extra_eval["2019_LA"]):,}')


  ASVspoof: 25380 samples, 20 speakers, rawboost=0.5
  ITW: 31779 samples (unlabeled returned)
  ITW: 31779 samples (labeled returned)
  ASVspoof Eval: 71237 samples
Source: 25,380 (20 speakers)
Target: 31,779, Eval: 31,779, 2019 LA: 71,237


In [ ]:
# Fix 1: PerGroupVIBModel — drop num_speakers kwarg from super().__init__
class PerGroupVIBModel(GLADAModel):
    def __init__(self, config):
        super().__init__(config)  # no num_speakers — trainer sets config.num_speakers
        K = config.num_groups
        D = config.hidden_dim
        bd = config.bottleneck_dim
        self.group_vibs = nn.ModuleList([VIBBottleneck(2 * D, bd) for _ in range(K)])
        self.group_compressed_proj = nn.Linear(K * bd, bd)
        if self.glada_disc is not None:
            self.glada_disc = GroupedLayerDomainDiscriminators(
                num_groups=K, input_dim=bd, hidden_dim=config.glada_disc_hidden,
            )

    def forward(self, input_values, attention_mask=None):
        backbone_frozen = not any(p.requires_grad for p in self.backbone.parameters())
        if backbone_frozen:
            with torch.no_grad():
                outputs = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        else:
            outputs = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        hidden_states = list(outputs.hidden_states)

        if attention_mask is not None:
            input_lengths = attention_mask.sum(dim=1)
            output_lengths = ((input_lengths - 1) // 320) + 1
            T = hidden_states[0].shape[1]
            frame_mask = (torch.arange(T, device=attention_mask.device).unsqueeze(0) < output_lengths.unsqueeze(1))
        else:
            frame_mask = None

        hidden_states = self.layer_norms(hidden_states)
        _, cka_loss, group_outputs = self.pooling(hidden_states, frame_mask=frame_mask)
        if not self.config.use_cka_loss:
            cka_loss = torch.tensor(0.0, device=group_outputs[0].device)

        z_groups = []
        kl_total = torch.tensor(0.0, device=group_outputs[0].device)
        for k, g_k in enumerate(group_outputs):
            z_k, kl_k = self.group_vibs[k](g_k, training=self.training)
            z_groups.append(z_k)
            kl_total = kl_total + kl_k
        kl_total = kl_total / max(len(group_outputs), 1)

        z = self.group_compressed_proj(torch.cat(z_groups, dim=-1))
        spoof_logits = self.task_classifier(z)
        result = {'spoof_logits': spoof_logits, 'kl_loss': kl_total, 'cka_loss': cka_loss, 'features': z}

        if self.glada_disc is not None:
            result['glada_domain_logits'] = self.glada_disc(z_groups, self.domain_grl_alpha)
            result['group_outputs'] = z_groups
        if self.domain_disc is not None:
            z_domain = GradientReversal.apply(z, self.domain_grl_alpha)
            result['domain_logits'] = self.domain_disc(z_domain)
        if self.speaker_adversary is not None:
            z_speaker = GradientReversal.apply(z, self.speaker_grl_alpha)
            result['speaker_logits'] = self.speaker_adversary(z_speaker)
        return result

MODEL_CLASSES = {'GLADAModel': GLADAModel, 'PerGroupVIBModel': PerGroupVIBModel}
print('PerGroupVIBModel redefined.')

PerGroupVIBModel redefined.


## 9. Custom Sweep Loop

Cannot use `glada.trainers.run_sweep` because we need to swap the model and trainer classes per variant. The loop below mirrors `run_sweep`'s incremental result-saving and GPU-cleanup pattern but lets us instantiate `PerGroupVIBModel` and `StaggeredAlphaTrainer` where appropriate.


In [ ]:
all_results = {}
sweep_start = time.time()
results_path = os.path.join(DRIVE_LOG_DIR, 'vib_stagger_results.json')

for i, (name, spec) in enumerate(VARIANTS.items()):
    print(f'\n{"="*60}\nVariant {i+1}/{len(VARIANTS)}: {name}\n{"="*60}')
    cfg = make_config(name)
    set_seed(cfg.seed)
    loaders = create_dataloaders(cfg, source_ds, target_ds, eval_ds, extra_eval)

    ModelCls = MODEL_CLASSES[spec['model_class']]
    TrainerCls = StaggeredAlphaTrainer if spec['staggered'] else GLADATrainer

    class _VariantTrainer(TrainerCls):
        def __init__(self, *a, **kw):
            super().__init__(*a, **kw)
            old_model = self.model
            new_model = ModelCls(self.config).to(self.device)  # ← FIX: no num_speakers
            old_state = old_model.state_dict()
            new_state = new_model.state_dict()
            for k in new_state.keys():
                if k in old_state and old_state[k].shape == new_state[k].shape:
                    new_state[k] = old_state[k]
            new_model.load_state_dict(new_state, strict=False)
            self.model = new_model
            self._setup_phase1()
            tr = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
            tot = sum(p.numel() for p in self.model.parameters())
            print(f'  [VARIANT] {ModelCls.__name__}; trainable={tr:,}/{tot:,}')

    try:
        trainer = _VariantTrainer(
            config=cfg,
            source_loader=loaders['source'], target_loader=loaders['target'],
            eval_loader=loaders['eval'],
            extra_eval_loaders={k: v for k, v in loaders.items() if k not in ('source', 'target', 'eval')},
            num_speakers=num_speakers,
        )
        all_results[name] = trainer.train()
    except Exception as e:
        import traceback; traceback.print_exc()
        all_results[name] = {'error': str(e)}

    save = {k: {kk: vv for kk, vv in v.items() if kk != 'history'} for k, v in all_results.items()}
    with open(results_path, 'w') as f:
        json.dump(save, f, indent=2, default=str)
    if 'trainer' in dir(): del trainer
    torch.cuda.empty_cache(); gc.collect()

print(f'\nSweep complete in {(time.time()-sweep_start)/60:.1f} min')


Variant 1/3: per_group_vib
  Updating num_speakers: 107 -> 20

Building GLADA model: wavlm-base-plus


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

  Trainable: 6,356,140 / 100,738,076
  Run dir: /content/drive/MyDrive/Jakob/Experiments/vib_stagger/per_group_vib/20260505_195057


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

  [VARIANT] PerGroupVIBModel; trainable=22,587,308/116,969,244

GLADA Training | wavlm-base-plus | seed=42
  Phase 1: 3 ep | Phase 2: 17 ep
  Loss Weights: D=0.1, KL=0.01, SPK=0.1, CKA=0.01
  KL Annealing: 0 -> 0.01 over 5 epochs
  Phase 2 Adv Ramp: 0.0 -> 1.0 over 3 P2 epochs (+CKA)


Ep 1:   0%|          | 0/396 [00:00<?, ?it/s]

... (62 outputs left out)

  [VARIANT] GLADAModel; trainable=6,356,140/100,738,076

GLADA Training | wavlm-base-plus | seed=42
  Phase 1: 3 ep | Phase 2: 17 ep
  Loss Weights: D=0.1, KL=0.01, SPK=0.1, CKA=0.01
  KL Annealing: 0 -> 0.01 over 5 epochs
  Phase 2 Adv Ramp: 0.0 -> 1.0 over 3 P2 epochs (+CKA)


Ep 1:   0%|          | 0/396 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Patched sweep — runs variants 2 and 3 only, skipping anything
# already in all_results.  Run AFTER interrupting the original
# sweep cell post-variant-1.
# ============================================================

# 1. Skip-set: anything that finished cleanly in the original loop
def _is_done(res):
    return isinstance(res, dict) and 'error' not in res and res.get('best_eer') is not None

remaining = {
    n: spec for n, spec in VARIANTS.items()
    if not _is_done(all_results.get(n, {}))
}
print(f'Already complete: {[n for n in VARIANTS if n not in remaining]}')
print(f'Will run:         {list(remaining)}')

# 2. Apply patches that should be active for the remaining variants
extra_eval.clear()                 # drop per-epoch 2019_LA eval
# (any other tweaks here, e.g. EVAL_BATCH_SIZE, alpha schedules, etc.)

# 3. Mirror the original loop body, iterating only `remaining`
sweep_resume_start = time.time()
for i, (name, spec) in enumerate(remaining.items()):
    elapsed = (time.time() - sweep_resume_start) / 60
    print(f'\n{"="*60}\nResume variant {i+1}/{len(remaining)}: {name}'
          f'\n  resume-elapsed: {elapsed:.1f} min\n{"="*60}')

    cfg = make_config(name)
    set_seed(cfg.seed)
    loaders = create_dataloaders(cfg, source_ds, target_ds, eval_ds, extra_eval)

    ModelCls   = MODEL_CLASSES[spec['model_class']]
    TrainerCls = StaggeredAlphaTrainer if spec['staggered'] else GLADATrainer

    class _VariantTrainer(TrainerCls):
        def __init__(self, *a, **kw):
            super().__init__(*a, **kw)
            old_state = self.model.state_dict()
            new_model = ModelCls(self.config).to(self.device)
            new_state = new_model.state_dict()
            for k in new_state:
                if k in old_state and old_state[k].shape == new_state[k].shape:
                    new_state[k] = old_state[k]
            new_model.load_state_dict(new_state, strict=False)
            self.model = new_model
            self._setup_phase1()

    try:
        trainer = _VariantTrainer(
            config=cfg,
            source_loader=loaders['source'],
            target_loader=loaders['target'],
            eval_loader=loaders['eval'],
            extra_eval_loaders={k: v for k, v in loaders.items()
                                if k not in ('source', 'target', 'eval')},
            num_speakers=num_speakers,
        )
        all_results[name] = trainer.train()
    except Exception as e:
        import traceback; traceback.print_exc()
        all_results[name] = {'error': str(e)}

    # Incremental save (omit history)
    save = {k: {kk: vv for kk, vv in v.items() if kk != 'history'}
            for k, v in all_results.items() if isinstance(v, dict)}
    with open(results_path, 'w') as f:
        json.dump(save, f, indent=2, default=str)

    if 'trainer' in dir(): del trainer
    torch.cuda.empty_cache(); gc.collect()

print(f'\nResume complete: {len(all_results)} total variants in all_results')

Already complete: ['per_group_vib']
Will run:         ['staggered', 'per_group_vib_staggered']

Resume variant 1/2: staggered
  resume-elapsed: 0.0 min
  Updating num_speakers: 107 -> 20

Building GLADA model: wavlm-base-plus


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

  Trainable: 6,356,140 / 100,738,076
  Run dir: /content/drive/MyDrive/Jakob/Experiments/vib_stagger/staggered/20260505_223808


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]


GLADA Training | wavlm-base-plus | seed=42
  Phase 1: 3 ep | Phase 2: 17 ep
  Loss Weights: D=0.1, KL=0.01, SPK=0.1, CKA=0.01
  KL Annealing: 0 -> 0.01 over 5 epochs
  Phase 2 Adv Ramp: 0.0 -> 1.0 over 3 P2 epochs (+CKA)


Ep 1:   0%|          | 0/396 [00:00<?, ?it/s]

... (118 outputs left out)

  Ep 19 P2: L=0.466 KL=10.00 | ITW=16.78%
  [STAGGER] P2 epoch 16: vib=True glada=True spk=True cka=True


Ep 20:   0%|          | 0/396 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


  [WARN] KL clamped 7000x consecutive (raw=3.511e+01, clamp=10.0)
  Ep 20 P2: L=0.466 KL=10.00 | ITW=16.74%

Done in 106.6 min | Best ITW EER: 14.24% (Ep 6)

Resume complete: 3 total variants in all_results


## 10. Decision Table

Compare the three variants against four reference points from `Coding/results.json`:
- `full_glada` (19.71%) — the negative-transfer config we're trying to fix
- `no_glada` (14.89%) — the single-DA baseline that beat full_glada
- `no_cka` (12.98%) — best wavlm-bp ablation
- `bb_wavlm-large_s42` (12.64%) — best overall (different backbone)


In [ ]:
REFERENCE = {
    'full_glada':         19.71,
    'no_glada':           14.89,
    'no_cka':             12.98,
    'bb_wavlm-large_s42': 12.64,
}

header = f'{"variant":<28s} {"ITW EER":>10s} {"vs full_glada":>15s} {"vs no_glada":>13s} {"vs no_cka":>11s}'
print(header); print('-' * len(header))
for name, res in all_results.items():
    if 'error' in res:
        print(f'{name:<28s} FAILED: {res["error"][:60]}'); continue
    eer = res.get('best_eer') or res.get('eval_metrics', {}).get('ITW', {}).get('eer')
    if eer is None:
        print(f'{name:<28s} NO EER'); continue
    d_full  = eer - REFERENCE['full_glada']
    d_no    = eer - REFERENCE['no_glada']
    d_nocka = eer - REFERENCE['no_cka']
    print(f'{name:<28s} {eer:>10.2f} {d_full:>+15.2f} {d_no:>+13.2f} {d_nocka:>+11.2f}')

print()
print('Reference (no retraining):')
for name, eer in REFERENCE.items():
    print(f'  {name:<28s} {eer:>10.2f}')

# --- Verdict ---
best_eer = min((res.get('best_eer') or 100) for res in all_results.values() if 'error' not in res)
best_var = [n for n, r in all_results.items() if 'error' not in r and r.get('best_eer') == best_eer]
print()
print('=' * 60)
if best_eer < REFERENCE['no_cka']:
    print(f'VERDICT: best variant ({best_var[0]} at {best_eer:.2f}%) BEATS no_cka ({REFERENCE["no_cka"]:.2f}%).')
    print('  -> The architectural fixes recover GLADA into competitive territory.')
    print('  -> Substantial defense / paper update.')
elif best_eer < REFERENCE['no_glada']:
    print(f'VERDICT: best variant ({best_var[0]} at {best_eer:.2f}%) beats no_glada ({REFERENCE["no_glada"]:.2f}%) but not no_cka.')
    print('  -> The fixes help — negative transfer was avoidable.')
    print('  -> Worth discussing in defense as a partial recovery.')
elif best_eer < REFERENCE['full_glada']:
    print(f'VERDICT: best variant ({best_var[0]} at {best_eer:.2f}%) beats full_glada ({REFERENCE["full_glada"]:.2f}%) but not no_glada.')
    print('  -> The fixes reduce the negative-transfer gap but do not close it.')
    print('  -> Defense talking point: the gradient-shock and uncompressed-disc hypotheses')
    print('     are partial explanations, not complete ones.')
else:
    print(f'VERDICT: best variant ({best_var[0]} at {best_eer:.2f}%) does NOT beat full_glada ({REFERENCE["full_glada"]:.2f}%).')
    print('  -> Neither architectural fix helps — negative-transfer claim is ROBUST.')
    print('  -> Defense talking point: VIB placement and gradient timing were not the binding constraint.')
print('=' * 60)


variant                         ITW EER   vs full_glada   vs no_glada   vs no_cka
---------------------------------------------------------------------------------
per_group_vib                     11.67           -8.04         -3.22       -1.31
staggered                         12.06           -7.65         -2.83       -0.92
per_group_vib_staggered           14.24           -5.47         -0.65       +1.26

Reference (no retraining):
  full_glada                        19.71
  no_glada                          14.89
  no_cka                            12.98
  bb_wavlm-large_s42                12.64

VERDICT: best variant (per_group_vib at 11.67%) BEATS no_cka (12.98%).
  -> The architectural fixes recover GLADA into competitive territory.
  -> Substantial defense / paper update.


In [ ]:
# ============================================================
# L1: best L0 variant + WavLM-large (RawBoost only)
# Queued after the L0 sweep. Reuses source_ds/target_ds/eval_ds/
# extra_eval (backbone-agnostic — raw audio), VARIANTS dict,
# MODEL_CLASSES, StaggeredAlphaTrainer, GLADATrainer, num_speakers.
# ============================================================

# ---- Knobs ----
L1_GATE_EER   = 13.0    # only fire L1 if best L0 ITW EER < this
L1_FORCE_RUN  = False   # set True to bypass the gate
L1_BATCH_SIZE = 24      # wavlm-large is ~316M params; A100 default

# ---- 1. Pick the winning L0 variant ----
valid_l0 = {
    n: r for n, r in all_results.items()
    if 'error' not in r and r.get('best_eer') is not None
}
if not valid_l0:
    print('L1 SKIPPED: no L0 variants completed. Inspect tracebacks above.')
    raise SystemExit(0)

best_name = min(valid_l0, key=lambda n: valid_l0[n]['best_eer'])
best_eer  = valid_l0[best_name]['best_eer']
best_spec = VARIANTS[best_name]
print(f'Best L0 variant: {best_name}')
print(f'  ITW EER  = {best_eer:.2f}%')
print(f'  model    = {best_spec["model_class"]}')
print(f'  stagger  = {best_spec["staggered"]}')

if best_eer >= L1_GATE_EER and not L1_FORCE_RUN:
    print(f'\nL1 SKIPPED: best L0 EER {best_eer:.2f}% >= gate {L1_GATE_EER:.2f}%.')
    print('Either L0 did not deliver, or set L1_FORCE_RUN=True to override.')
    raise SystemExit(0)

# ---- 2. L1 config: same schedule, larger backbone, smaller batch ----
L1_DRIVE_LOG_DIR = '/content/drive/MyDrive/Jakob/Experiments/vib_stagger_L1'
os.makedirs(L1_DRIVE_LOG_DIR, exist_ok=True)

l1_cfg = GLADAConfig(
    experiment_name=f'L1_{best_name}_wavlm-large',
    backbone='wavlm-large',
    seed=42,
    batch_size=L1_BATCH_SIZE,
    num_workers=0,
    phase1_epochs=PHASE1,
    phase2_epochs=PHASE2,
    drive_log_dir=L1_DRIVE_LOG_DIR,
    run_id=time.strftime('%Y%m%d_%H%M%S'),
)
set_seed(l1_cfg.seed)

# Datasets are backbone-agnostic (raw audio @ 16kHz) — only rebuild loaders.
l1_loaders = create_dataloaders(l1_cfg, source_ds, target_ds, eval_ds, extra_eval)

# ---- 3. Build trainer with the winning variant's model + schedule ----
ModelCls   = MODEL_CLASSES[best_spec['model_class']]
TrainerCls = StaggeredAlphaTrainer if best_spec['staggered'] else GLADATrainer

class _L1Trainer(TrainerCls):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        # Same model-swap pattern as the L0 _VariantTrainer:
        # parent built self.model = GLADAModel(cfg); replace with the
        # chosen variant class and rebuild the Phase 1 optimizer so it
        # points at the new params.
        old_state = self.model.state_dict()
        new_model = ModelCls(self.config).to(self.device)
        new_state = new_model.state_dict()
        for k in new_state:
            if k in old_state and old_state[k].shape == new_state[k].shape:
                new_state[k] = old_state[k]
        new_model.load_state_dict(new_state, strict=False)
        self.model = new_model
        self._setup_phase1()
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.model.parameters())
        print(f'  [L1] {ModelCls.__name__} on wavlm-large; '
              f'trainable={trainable:,}/{total:,}')

# ---- 4. Run L1 ----
l1_results_path = os.path.join(L1_DRIVE_LOG_DIR, 'L1_wavlm_large_results.json')
l1_start = time.time()
print(f'\n{"="*60}\nL1: {best_name} + wavlm-large + RawBoost\n{"="*60}')

try:
    l1_trainer = _L1Trainer(
        config=l1_cfg,
        source_loader=l1_loaders['source'],
        target_loader=l1_loaders['target'],
        eval_loader=l1_loaders['eval'],
        extra_eval_loaders={k: v for k, v in l1_loaders.items()
                            if k not in ('source', 'target', 'eval')},
        num_speakers=num_speakers,
    )
    l1_results = l1_trainer.train()
except Exception as e:
    import traceback; traceback.print_exc()
    l1_results = {'error': str(e)}

# ---- 5. Save + report ----
if 'error' in l1_results:
    l1_save = dict(l1_results)
else:
    l1_save = {k: v for k, v in l1_results.items() if k != 'history'}
l1_save['source_l0_variant'] = best_name
l1_save['source_l0_eer']     = best_eer

with open(l1_results_path, 'w') as f:
    json.dump(l1_save, f, indent=2, default=str)

print(f'\n{"="*60}')
print(f'L1 complete in {(time.time()-l1_start)/60:.1f} min')
l1_eer = l1_results.get('best_eer')
if isinstance(l1_eer, (int, float)):
    print(f'  L1 ({best_name} + wavlm-large)       : {l1_eer:.2f}%')
    print(f'  L0 ({best_name} + wavlm-base-plus)   : {best_eer:.2f}%')
    print(f'  delta vs L0                          : {l1_eer - best_eer:+.2f} pp')
    print(f'  delta vs project SOTA (12.64%)       : {l1_eer - 12.64:+.2f} pp')
    print(f'  delta vs full_glada wavlm-large      : {l1_eer - 12.64:+.2f} pp')
else:
    print(f'  L1 FAILED: {l1_results.get("error", "unknown")}')
print('='*60)

if 'l1_trainer' in dir(): del l1_trainer
torch.cuda.empty_cache(); gc.collect()

Best L0 variant: per_group_vib
  ITW EER  = 11.67%
  model    = PerGroupVIBModel
  stagger  = False

L1: per_group_vib + wavlm-large + RawBoost
  Updating num_speakers: 107 -> 20

Building GLADA model: wavlm-large


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

  Trainable: 12,975,163 / 328,428,283
  Run dir: /content/drive/MyDrive/Jakob/Experiments/vib_stagger_L1/L1_per_group_vib_wavlm-large/20260506_021047


... (47 outputs left out)

Ep 16:   0%|          | 0/1057 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


  Ep 16 P2: L=0.370 KL=0.33 | ITW=26.16%
  Early stopping at epoch 16

Done in 161.7 min | Best ITW EER: 13.52% (Ep 1)

L1 complete in 162.1 min
  L1 (per_group_vib + wavlm-large)       : 13.52%
  L0 (per_group_vib + wavlm-base-plus)   : 11.67%
  delta vs L0                          : +1.85 pp
  delta vs project SOTA (12.64%)       : +0.88 pp
  delta vs full_glada wavlm-large      : +0.88 pp


149

In [ ]:
# ============================================================
# L1 Eval Cell A: Load best L1 checkpoint
# ============================================================
from glada.datasets.asvspoof2021 import ASVspoof2021Dataset
from glada.datasets.wavefake     import WaveFakeDataset
from glada.datasets.codecfake    import CodecFakeDataset
from glada.evaluation.evaluator  import MultiDatasetEvaluator
from torch.utils.data import DataLoader
import shutil, tarfile, glob, subprocess, warnings, logging

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

# Locate the L1 run directory.
# train() returns {'best_eer','best_epoch','run_dir','history'} → use it.
if 'l1_results' in dir() and isinstance(l1_results, dict) and 'run_dir' in l1_results:
    L1_RUN_DIR = l1_results['run_dir']
else:
    # Fallback if kernel restarted between training and eval
    pat = os.path.join(L1_DRIVE_LOG_DIR, '*', '*')
    candidates = sorted(glob.glob(pat))
    assert candidates, f'No L1 runs found under {L1_DRIVE_LOG_DIR}'
    L1_RUN_DIR = candidates[-1]

L1_BEST_CKPT = os.path.join(L1_RUN_DIR, 'best_model.pt')
print(f'L1 run dir: {L1_RUN_DIR}')
print(f'Loading:    {L1_BEST_CKPT}')

# Rebuild model with the same variant class used during L1 training.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
eval_model = ModelCls(l1_cfg).to(device)

ckpt = torch.load(L1_BEST_CKPT, map_location='cpu', weights_only=False)
state = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt
missing, unexpected = eval_model.load_state_dict(state, strict=False)
if missing:    print(f'  missing keys (expected for some heads): {len(missing)}')
if unexpected: print(f'  unexpected keys: {len(unexpected)}')
eval_model.eval()

trainable = sum(p.numel() for p in eval_model.parameters())
print(f'Eval model: {ModelCls.__name__} on {l1_cfg.backbone} ({trainable/1e6:.1f}M params)')

/content/glada/evaluation/evaluator.py:194: SyntaxWarning: invalid escape sequence '\p'
  it to ``\pm`` or the backslash will be lost in the output string.


L1 run dir: /content/drive/MyDrive/Jakob/Experiments/vib_stagger_L1/L1_per_group_vib_wavlm-large/20260506_021047
Loading:    /content/drive/MyDrive/Jakob/Experiments/vib_stagger_L1/L1_per_group_vib_wavlm-large/20260506_021047/best_model.pt


Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

Eval model: PerGroupVIBModel on wavlm-large (365.5M params)


In [ ]:
# ============================================================
# L1 Eval Cell B: Dataset registry + extract/cleanup helpers
# 6 datasets: ITW, 2019_LA, 2021_LA, 2021_DF, WaveFake, CodecFake+
# Sequential extract → eval → cleanup so we never exceed ~70GB SSD.
# ============================================================
LOCAL_ROOT      = '/content/data'
DRIVE_BASE      = '/content/drive/MyDrive/Jakob/Datasets'
CODECFAKE_ROOT  = '/content/drive/Shareddrives/Jakob-298-datasets/CodecFakePlus'
EVAL_BATCH_SIZE = 16  # wavlm-large + no-grad eval; raise to 32 if A100 has headroom

def _extract(tar_path, extract_to, name):
    if not os.path.exists(tar_path):
        print(f'  [{name}] archive missing: {tar_path}'); return False
    os.makedirs(extract_to, exist_ok=True)
    if os.listdir(extract_to):
        print(f'  [{name}] already extracted at {extract_to}'); return True
    mode = 'r:gz' if tar_path.endswith('.gz') else 'r:*'
    print(f'  [{name}] extracting {tar_path} ...')
    with tarfile.open(tar_path, mode) as tf: tf.extractall(extract_to)
    return True

def _extract_multipart(part_glob, extract_to, name):
    parts = sorted(glob.glob(part_glob))
    if not parts:
        print(f'  [{name}] no parts at {part_glob}'); return False
    os.makedirs(extract_to, exist_ok=True)
    if os.listdir(extract_to):
        print(f'  [{name}] already extracted at {extract_to}'); return True
    cat_path = os.path.join(LOCAL_ROOT, f'{name}_concat.tar.gz')
    print(f'  [{name}] concatenating {len(parts)} parts ...')
    with open(cat_path, 'wb') as out:
        for p in parts:
            with open(p, 'rb') as inp: shutil.copyfileobj(inp, out)
    print(f'  [{name}] extracting concatenated archive ...')
    with tarfile.open(cat_path, 'r:gz') as tf: tf.extractall(extract_to)
    os.remove(cat_path)
    return True

def _cleanup(local_dir, name):
    if local_dir and os.path.exists(local_dir):
        shutil.rmtree(local_dir); print(f'  [{name}] cleaned up {local_dir}')

def _find(base_dir, *patterns):
    for pat in patterns:
        m = glob.glob(os.path.join(base_dir, pat), recursive=True)
        if m: return m[0]
    return None

# Loader factories (lazy — datasets only built when we reach that row)
def _itw_loader():
    return DataLoader(
        InTheWildDataset(ITW_META, ITW_AUDIO, config=l1_cfg, use_labels=True),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

def _asv19_loader():
    return DataLoader(
        ASVspoofEvalDataset(ASV_EVAL_PROTO, ASV_EVAL_AUDIO, config=l1_cfg),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

def _asv21_loader(local_dir, track):
    keys  = _find(local_dir, '**/trial_metadata.txt', f'**/keys/{track}/CM/trial_metadata.txt')
    audio = _find(local_dir, f'**/ASVspoof2021_{track}_eval/flac', '**/flac')
    assert keys and audio, f'{track}: keys={keys} audio={audio}'
    print(f'  keys:  {keys}\n  audio: {audio}')
    return DataLoader(
        ASVspoof2021Dataset(keys, audio, max_len=l1_cfg.max_audio_len),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

def _wavefake_loader():
    return DataLoader(
        WaveFakeDataset('/content/data/wavefake', max_len=l1_cfg.max_audio_len),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

def _codecfake_loader():
    return DataLoader(
        CodecFakeDataset(CODECFAKE_ROOT, max_len=l1_cfg.max_audio_len),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# Registry — order chosen so the cheapest / already-extracted ones run first
DATASET_REGISTRY = [
    # Already on SSD from training-setup cell — no extract / no cleanup
    {'name': 'ITW',      'extract': None, 'local_dir': None, 'loader': _itw_loader},
    {'name': '2019_LA',  'extract': None, 'local_dir': None, 'loader': _asv19_loader},
    # Drive-only archives — extract to SSD, eval, then cleanup
    {'name': '2021_LA',
     'extract': lambda d: _extract(os.path.join(DRIVE_BASE, 'ASVspoof2021_LA_eval.tar.gz'), d, '2021_LA'),
     'local_dir': os.path.join(LOCAL_ROOT, 'asvspoof21_la'),
     'loader':    lambda d=None: _asv21_loader(os.path.join(LOCAL_ROOT, 'asvspoof21_la'), 'LA')},
    {'name': '2021_DF',
     'extract': lambda d: _extract_multipart(
         os.path.join(DRIVE_BASE, 'ASVspoof2021_DF', 'ASVspoof2021_DF_eval_part_*'), d, '2021_DF'),
     'local_dir': os.path.join(LOCAL_ROOT, 'asvspoof21_df'),
     'loader':    lambda d=None: _asv21_loader(os.path.join(LOCAL_ROOT, 'asvspoof21_df'), 'DF')},
    {'name': 'WaveFake',
     'extract': lambda d: _extract(os.path.join(DRIVE_BASE, 'WaveFake.tar.gz'), d, 'WaveFake'),
     'local_dir': os.path.join(LOCAL_ROOT, 'wavefake'),
     'loader':    _wavefake_loader},
    # Streamed directly from Shared Drive — no extraction, no cleanup
    {'name': 'CodecFakePlus', 'extract': None, 'local_dir': None, 'loader': _codecfake_loader},
]

print(f'Registered {len(DATASET_REGISTRY)} eval datasets.')

Registered 6 eval datasets.


In [ ]:
# ============================================================
# L1 Eval Cell C: extract → eval → cleanup → repeat
# Saves per-dataset EER + 95% bootstrap CI to <run_dir>/multidataset_eval.json
# ============================================================
evaluator = MultiDatasetEvaluator(eval_model, l1_cfg, device, eval_loaders={})
multi_results = {}
eval_start = time.time()

print('=' * 72)
print(f'L1 multi-dataset eval: {best_name} + {l1_cfg.backbone}')
print('=' * 72)

for entry in DATASET_REGISTRY:
    name = entry['name']
    print(f'\n--- {name} ---')

    # 1. Extract if needed
    if entry['extract'] is not None:
        ok = entry['extract'](entry['local_dir'])
        if not ok:
            print(f'  SKIP {name}'); continue

    # 2. Build loader + evaluate
    try:
        loader = entry['loader']()
        print(f'  samples: {len(loader.dataset):,}')
        result = evaluator.evaluate_dataset(loader, name)
        multi_results[name] = result
        eer  = result.get('eer', float('nan'))
        ci_l = result.get('ci_low', float('nan'))
        ci_h = result.get('ci_high', float('nan'))
        print(f'  EER: {eer:.2f}%  [95% CI {ci_l:.2f}–{ci_h:.2f}]')
    except Exception as e:
        import traceback; traceback.print_exc()
        multi_results[name] = {'error': str(e)}

    # 3. Cleanup + free GPU memory
    if 'loader' in dir(): del loader
    torch.cuda.empty_cache(); gc.collect()
    if entry.get('local_dir'):
        _cleanup(entry['local_dir'], name)

    # Disk usage check
    df = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
    print(f'  disk: {df.stdout.strip().splitlines()[-1]}')

# --- Persist + summarize ---
out_path = os.path.join(L1_RUN_DIR, 'multidataset_eval.json')
payload = {
    'backbone': l1_cfg.backbone,
    'variant':  best_name,
    'model_class': ModelCls.__name__,
    'staggered': best_spec['staggered'],
    'l0_source_eer': best_eer,
    'results': {k: {kk: (float(vv) if hasattr(vv,'item') else vv)
                    for kk, vv in v.items()} for k, v in multi_results.items()},
}
with open(out_path, 'w') as f:
    json.dump(payload, f, indent=2, default=str)
print(f'\nSaved: {out_path}')

# --- Pretty table ---
print('\n' + '=' * 72)
print(f'L1 multi-dataset summary — {best_name} on {l1_cfg.backbone}')
print('=' * 72)
print(f'{"Dataset":<16} {"EER (%)":>10} {"95% CI":>20} {"AUC":>8} {"N":>10}')
print('-' * 72)
for name, r in multi_results.items():
    if 'error' in r:
        print(f'{name:<16} {"FAIL":>10} {r["error"][:30]}'); continue
    eer  = r.get('eer', float('nan'))
    ci_l = r.get('ci_low', float('nan'))
    ci_h = r.get('ci_high', float('nan'))
    auc  = r.get('auc',   float('nan'))
    n    = r.get('n_samples', 0)
    print(f'{name:<16} {eer:>9.2f}  [{ci_l:>5.2f}–{ci_h:>5.2f}]  {auc:>7.4f}  {n:>10,}')
print('-' * 72)
print(f'Total eval time: {(time.time()-eval_start)/60:.1f} min')

L1 multi-dataset eval: per_group_vib + wavlm-large

--- ITW ---
  ITW: 31779 samples (labeled returned)
  samples: 31,779


Eval ITW:   0%|          | 0/1987 [00:00<?, ?it/s]

  EER: 13.51%  [95% CI 13.14–13.90]
  disk: overlay         236G   93G  143G  40% /

--- 2019_LA ---
  ASVspoof Eval: 71237 samples
  samples: 71,237


Eval 2019_LA:   0%|          | 0/4453 [00:00<?, ?it/s]

  EER: 0.98%  [95% CI 0.90–1.06]
  disk: overlay         236G   93G  143G  40% /

--- 2021_LA ---
  [2021_LA] extracting /content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_LA_eval.tar.gz ...


Traceback (most recent call last):
  File "/tmp/ipykernel_452/1055185103.py", line 25, in <cell line: 0>
    loader = entry['loader']()
             ^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_452/3350962338.py", line 88, in <lambda>
    'loader':    lambda d=None: _asv21_loader(os.path.join(LOCAL_ROOT, 'asvspoof21_la'), 'LA')},
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_452/3350962338.py", line 63, in _asv21_loader
    assert keys and audio, f'{track}: keys={keys} audio={audio}'
           ^^^^^^^^^^^^^^
AssertionError: LA: keys=None audio=/content/data/asvspoof21_la/ASVspoof2021_LA_eval/flac


  [2021_LA] cleaned up /content/data/asvspoof21_la
  disk: overlay         236G   93G  143G  40% /

--- 2021_DF ---
  [2021_DF] no parts at /content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_DF/ASVspoof2021_DF_eval_part_*
  SKIP 2021_DF

--- WaveFake ---
  [WaveFake] archive missing: /content/drive/MyDrive/Jakob/Datasets/WaveFake.tar.gz
  SKIP WaveFake

--- CodecFakePlus ---
  CodecFake+: 0 samples (0 bonafide, 0 spoof)
  samples: 0


Eval CodecFakePlus: 0it [00:00, ?it/s]

  EER: 50.00%  [95% CI 50.00–50.00]
  disk: overlay         236G   93G  143G  40% /

Saved: /content/drive/MyDrive/Jakob/Experiments/vib_stagger_L1/L1_per_group_vib_wavlm-large/20260506_021047/multidataset_eval.json

L1 multi-dataset summary — per_group_vib on wavlm-large
Dataset             EER (%)               95% CI      AUC          N
------------------------------------------------------------------------
ITW                  13.51  [13.14–13.90]   0.9392      31,779
2019_LA               0.98  [ 0.90– 1.06]   0.9980      71,237
2021_LA                FAIL LA: keys=None audio=/content/d
CodecFakePlus        50.00  [50.00–50.00]   0.5000           0
------------------------------------------------------------------------
Total eval time: 12.0 min
